<a href="https://colab.research.google.com/github/jacobhroutzong/Reinforcement-Learning/blob/main/Running_Olmo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Upgrade to latest transformers (v4.57+ required for OLMo 3 native support)
!pip install --upgrade transformers accelerate -q

print("✅ Packages ready — OLMo 3 supported natively!")

✅ Packages ready — OLMo 3 supported natively!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Official ID for the helpful chat model
model_id = "allenai/Olmo-3-7B-Instruct"

print(f"\n🔄 Loading {model_id}... (~14GB download first time)")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("🎉 OLMo 3 7B Instruct loaded!")


🔄 Loading allenai/Olmo-3-7B-Instruct... (~14GB download first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

🎉 OLMo 3 7B Instruct loaded!


In [ ]:
# Set pad_token to eos_token (silences the warning)
tokenizer.pad_token = tokenizer.eos_token

messages = [
    {"role": "user", "content": "In one clear paragraph, explain why OLMo is considered the most open large language model family in 2025."}
]

# Apply the official chat template (ChatML-style with <|im_start|> tags)
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

output = model.generate(
    inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id  # Explicitly set to avoid any issues
)

response = tokenizer.decode(output[0], skip_special_tokens=True)

# Extract just the assistant's reply
assistant_start = response.rfind("<|im_start|>assistant")
if assistant_start != -1:
    print("\n🤖 OLMo 3 Instruct response:")
    print(response[assistant_start + len("<|im_start|>assistant"):].strip())
else:
    print(response)

system
You are a helpful function-calling AI assistant. You do not currently have access to any functions. <functions></functions>
user
In one clear paragraph, explain why OLMo is considered the most open large language model family in 2025.
assistant
OLMo is widely regarded as the most open large language model (LLM) family in 2025 because its development is characterized by maximal transparency and accessibility. Unlike many proprietary models whose weights, architectures, and training details are kept closed, OLMo releases its complete training codebase, model weights, and detailed documentation, allowing researchers and the public to reproduce, modify, and build upon the models. This open approach not only fosters collaborative innovation but also enables broader evaluation and ethical scrutiny, setting a new standard for openness in the field and contrasting sharply with the more closed or partially open alternatives from other major projects.


In [ ]:
# Super-simple chat function for OLMo 3 Instruct (and future models)
def chat(prompt, max_tokens=300, temperature=0.7):
    tokenizer.pad_token = tokenizer.eos_token  # Silences warnings

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    output = model.generate(
        inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Cleanly extract just the assistant's reply
    if "<|im_start|>assistant" in full_text:
        return full_text.split("<|im_start|>assistant")[-1].strip()
    return full_text

print("✅ Chat helper installed! Now you can just do: chat('your prompt')")

✅ Chat helper installed! Now you can just do: chat('your prompt')


In [ ]:
response = chat("Tell me a fun fact about the Allen Institute for AI (AI2).")

print("\n🤖 OLMo 3 says:")
print(response)


🤖 OLMo 3 says:
system
You are a helpful function-calling AI assistant. You do not currently have access to any functions. <functions></functions>
user
Tell me a fun fact about the Allen Institute for AI (AI2).
assistant
Sure! One fun fact about the Allen Institute for Artificial Intelligence (AI2) is that it was founded in 2008 by Paul Allen, the late co-founder of Microsoft, who also funded the Seattle-based institute to advance artificial general intelligence (AGI) and conduct open, collaborative research. The institute is known for its open access publications—many of its research papers are published freely online, which is quite rare in the often-guarded world of scientific publishing. This commitment to openness is part of what makes AI2 unique in the AI research community!
